In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy.stats import truncnorm

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "导入 pandas 失败。请使用与当前 NumPy 兼容的 pandas，推荐 Python 3.9 + numpy<2。"
    ) from exc

try:
    from astropy.cosmology import Planck18 as cosmo
except Exception as exc:
    raise RuntimeError("导入 astropy 失败，请先安装 astropy。") from exc

CANDIDATE_DIRS = [Path.cwd(), Path.cwd() / "tianqin_dc" / "codes"]
NOTEBOOK_DIR = next((path for path in CANDIDATE_DIRS if (path / "z_to_D.py").exists()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError(
        "未找到 z_to_D.py。请从仓库根目录或 tianqin_dc/codes 目录运行该 notebook。"
    )

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from z_to_D import z_to_Dz

RANDOM_SEED = None  # 设为整数可复现结果
N_CATALOGS = 1
Z_MAX = 2.8
OBSERVATION_TIME_YR = 5.0
MIN_SOURCES = 20
TARGET_SOURCES = 100_000  # 设为 None 时按并合率泊松抽样；设为整数时固定生成该数量

COMOVING_DISTANCE_GPC = z_to_Dz(Z_MAX) / 1e6
# 以 z_max 对应的共动距离为半径，估算球形观测体积
SPHERICAL_VOLUME_GPC3 = 4.0 / 3.0 * np.pi * COMOVING_DISTANCE_GPC**3
OUTPUT_DIR = NOTEBOOK_DIR / "bp2p_bbh_catalogs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

def create_skewed_gaussian(mean, upper_err, lower_err, size=1):
    """创建偏高斯分布，根据上下误差不对称。"""
    if upper_err == lower_err:
        return np.random.normal(mean, upper_err, size)

    scale = (upper_err + lower_err) / 2
    skew_factor = (upper_err - lower_err) / (upper_err + lower_err)
    samples = truncnorm.rvs(-2, 2, loc=mean + skew_factor, scale=scale, size=size)
    return np.clip(samples, mean - lower_err, mean + upper_err)

def power_law_norm(m_min, m_max, alpha):
    if np.isclose(alpha, 0.0):
        return np.log(m_max / m_min)
    return (m_min ** (-alpha) - m_max ** (-alpha)) / alpha

def sample_power_law_segment(size, m_min, m_max, alpha):
    if size == 0:
        return np.array([], dtype=float)

    u = np.random.uniform(0, 1, size)
    if np.isclose(alpha, 0.0):
        log_m = np.log(m_min) + u * (np.log(m_max) - np.log(m_min))
        return np.exp(log_m)

    return (m_min ** (-alpha) + u * (m_max ** (-alpha) - m_min ** (-alpha))) ** (-1 / alpha)

def sample_m1(n_sources, alpha1, alpha2, mu1, mu2, sigma1=0.3, sigma2=3.0):
    """根据给定参数采样主质量。"""
    lambda0, lambda1, lambda2 = 0.324, 0.634, 0.042
    comp_choice = np.random.choice([0, 1, 2], size=n_sources, p=[lambda0, lambda1, lambda2])

    m1_samples = np.zeros(n_sources)
    m_break = 20.0
    m_low, m_high = 5.0, 100.0

    mask_bpl = comp_choice == 0
    n_bpl = mask_bpl.sum()
    if n_bpl > 0:
        A_low = power_law_norm(m_low, m_break, alpha1)
        A_high = power_law_norm(m_break, m_high, alpha2)
        frac_low = A_low / (A_low + A_high)

        low_mask = np.random.uniform(0, 1, n_bpl) < frac_low
        bpl_indices = np.where(mask_bpl)[0]
        low_indices = bpl_indices[low_mask]
        high_indices = bpl_indices[~low_mask]

        m1_samples[low_indices] = sample_power_law_segment(len(low_indices), m_low, m_break, alpha1)
        m1_samples[high_indices] = sample_power_law_segment(len(high_indices), m_break, m_high, alpha2)

    mask_g1 = comp_choice == 1
    if mask_g1.sum() > 0:
        m1_samples[mask_g1] = truncnorm.rvs(
            (m_low - mu1) / sigma1,
            (m_high - mu1) / sigma1,
            loc=mu1,
            scale=sigma1,
            size=mask_g1.sum(),
        )

    mask_g2 = comp_choice == 2
    if mask_g2.sum() > 0:
        m1_samples[mask_g2] = truncnorm.rvs(
            (m_low - mu2) / sigma2,
            (m_high - mu2) / sigma2,
            loc=mu2,
            scale=sigma2,
            size=mask_g2.sum(),
        )

    return m1_samples

def generate_redshift_distribution(n_sources, kappa, z_max=Z_MAX):
    z_values = np.linspace(0, z_max, 5000)
    dVcdz = cosmo.differential_comoving_volume(z_values).value
    pz = (1 + z_values) ** (kappa - 1) * dVcdz
    pz = pz / np.trapezoid(pz, z_values)

    dz = z_values[1] - z_values[0]
    cdf = np.cumsum(pz) * dz
    cdf = cdf / cdf[-1]

    u = np.random.rand(n_sources)
    return np.interp(u, cdf, z_values)

def sample_q(m1, beta_q, m2_low=3.0):
    q_min = m2_low / m1
    q_max = 1.0
    u = np.random.uniform(0, 1, len(m1))
    exponent = beta_q + 1

    if np.isclose(exponent, 0.0):
        log_q = np.log(q_min) + u * (np.log(q_max) - np.log(q_min))
        return np.exp(log_q)

    return (q_min**exponent + u * (q_max**exponent - q_min**exponent)) ** (1 / exponent)

def generate_catalog_from_rate(catalog_id, observation_time, volume, min_sources, target_sources=None):
    """根据并合率计算源数量并生成 catalog。"""
    alpha1 = create_skewed_gaussian(1.7, 1.2, 1.8)[0]
    alpha2 = create_skewed_gaussian(4.5, 1.6, 1.3)[0]
    beta_q = create_skewed_gaussian(1.2, 1.2, 1.0)[0]
    mu1 = create_skewed_gaussian(9.8, 0.3, 0.6)[0]
    mu2 = create_skewed_gaussian(32.7, 2.7, 6.5)[0]
    kappa = create_skewed_gaussian(3.2, 0.94, 1.00)[0]

    merger_rate = np.random.uniform(14, 26)
    expected_n = merger_rate * volume * observation_time
    if target_sources is None:
        n_sources = max(np.random.poisson(expected_n), min_sources)
    else:
        n_sources = int(target_sources)
        if n_sources <= 0:
            raise ValueError("target_sources must be positive when set.")

    print(
        f"Catalog {catalog_id:2d}: 并合率={merger_rate:.1f}, 共动体积={volume:.3f} Gpc³, 时间={observation_time:.1f} yr"
    )
    if target_sources is None:
        print(f"               预期源数={expected_n:.1f}, 实际源数={n_sources}")
    else:
        print(f"               预期源数={expected_n:.1f}, 目标源数={n_sources}")
    print(f"               参数: alpha1={alpha1:.2f}, alpha2={alpha2:.2f}, beta_q={beta_q:.2f}, kappa={kappa:.2f}")

    z_samples = generate_redshift_distribution(n_sources, kappa)
    m1_samples = sample_m1(n_sources, alpha1, alpha2, mu1, mu2)
    q_samples = sample_q(m1_samples, beta_q)
    m2_samples = q_samples * m1_samples
    t_c_samples = np.random.uniform(0, observation_time, n_sources)
    psi_samples = np.random.uniform(0, np.pi, n_sources)

    df = pd.DataFrame(
        {
            "z": z_samples,
            "m1_Msun": m1_samples,
            "m2_Msun": m2_samples,
            "t_c_yr": t_c_samples,
            "psi_rad": psi_samples,
        }
    )

    return df, {
        "alpha1": alpha1,
        "alpha2": alpha2,
        "beta_q": beta_q,
        "mu1": mu1,
        "mu2": mu2,
        "kappa": kappa,
        "merger_rate": merger_rate,
        "N_sources": n_sources,
        "observation_time": observation_time,
        "volume": volume,
        "expected_N": expected_n,
    }

def format_stat(series, decimals):
    if len(series) == 1:
        return f"{series.iloc[0]:.{decimals}f} (单个 catalog，无样本方差)"
    return f"{series.mean():.{decimals}f} ± {series.std():.{decimals}f}"

catalog_params = []
total_sources = 0

print(f"根据并合率生成 {N_CATALOGS} 个双黑洞 catalog (CSV 格式):")
print(f"z_max = {Z_MAX:.2f}, D_c(z_max) = {COMOVING_DISTANCE_GPC:.3f} Gpc, 体积 = {SPHERICAL_VOLUME_GPC3:.3f} Gpc³")
if RANDOM_SEED is not None:
    print(f"随机种子: {RANDOM_SEED}")
print(f"输出目录: {OUTPUT_DIR.resolve()}")
print("=" * 80)

for i in range(N_CATALOGS):
    df, params = generate_catalog_from_rate(
        catalog_id=i + 1,
        observation_time=OBSERVATION_TIME_YR,
        volume=SPHERICAL_VOLUME_GPC3,
        min_sources=MIN_SOURCES,
        target_sources=TARGET_SOURCES,
    )

    filename = OUTPUT_DIR / f"catalog_{i + 1:03d}.csv"
    df.to_csv(filename, index=False)

    catalog_params.append(params)
    total_sources += params["N_sources"]

param_summary = pd.DataFrame(catalog_params)
param_summary.to_csv(OUTPUT_DIR / "catalog_parameters_summary.csv", index=False)

print("=" * 80)
print(f"生成完成：共 {N_CATALOGS} 个 catalog")
print(f"总源数量: {total_sources}")
print(f"平均每个 catalog: {total_sources / N_CATALOGS:.1f} 个源")
print(f"最小 catalog: {min(p['N_sources'] for p in catalog_params)} 个源")
print(f"最大 catalog: {max(p['N_sources'] for p in catalog_params)} 个源")
print(f"输出目录: {OUTPUT_DIR.resolve()}/")

print("\n参数统计:")
print(f"alpha1: {format_stat(param_summary['alpha1'], 2)}")
print(f"alpha2: {format_stat(param_summary['alpha2'], 2)}")
print(f"beta_q: {format_stat(param_summary['beta_q'], 2)}")
print(f"mu1: {format_stat(param_summary['mu1'], 1)}")
print(f"mu2: {format_stat(param_summary['mu2'], 1)}")
print(f"kappa: {format_stat(param_summary['kappa'], 2)}")
print(f"并合率: {format_stat(param_summary['merger_rate'], 1)} Gpc-3 yr-1")
print(f"观测时间: {format_stat(param_summary['observation_time'], 1)} 年")
print(f"观测体积: {format_stat(param_summary['volume'], 1)} Gpc³")

print("\n第一个 catalog 的前 5 行示例:")
try:
    example_df = pd.read_csv(OUTPUT_DIR / "catalog_001.csv")
    print(example_df.head())
except Exception as exc:
    print(f"读取示例文件失败: {exc}")
